<a href="https://colab.research.google.com/github/toanlion24/Vietnamese-license-plates-_-Group-3---NEW/blob/main/train_qwen2vl_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune Qwen2-VL-2B-Instruct cho VN License Plate Recognition

## Giới thiệu

Colab này fine-tune **Qwen2-VL-2B-Instruct** cho nhận diện biển số xe Việt Nam sử dụng:
- **Unsloth**: Giảm VRAM, tăng tốc training
- **QLoRA**: 4-bit quantization để train trên GPU giới hạn

## Runtime Setup

- **Runtime type**: GPU (T4, A100)
- **RAM**: High RAM để xử lý image crops

## Thời gian ước tính

- 100 samples, 3 epochs: ~10-15 phút (T4)
- 500 samples, 3 epochs: ~30-45 phút (T4)
- 1000 samples, 3 epochs: ~60-90 phút (A100)


# 1. Cài đặt thư viện

In [ ]:
# Cài đặt các phiên bản ổn định để tránh xung đột với Unsloth
!pip install --quiet --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --upgrade "transformers<4.47.0" "datasets<3.0.0" "trl<0.13.0" "accelerate<1.1.0"
!pip install --quiet qwen-vl-utils

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 108.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.5.4 requires datasets!=4.0.*,!

# 2. Xác thực Hugging Face

In [ ]:
from google.colab import userdata

# Lấy token từ Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError("❌ Chưa có HF_TOKEN! Vào Runtime > Manage secrets và thêm HF_TOKEN")

# Login Hugging Face
!huggingface-cli login --token {HF_TOKEN}

print("✅ Đăng nhập Hugging Face thành công!")


Hint: A new version of huggingface_hub (1.17.0) is available! You are using version 1.16.1.
To update, run: hf update
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help

✅ Đăng nhập Hugging Face thành công!


# 3. Upload Data

## Cách 1: Upload trực tiếp (cho dataset nhỏ < 100MB)

In [ ]:
from google.colab import files
import os

# Tạo thư mục data
!mkdir -p /content/vn_plate_data

print("📤 Upload file manifest CSV (image_id, image_path, text_gt)")
uploaded = files.upload()

# Đọc manifest
import pandas as pd
manifest = pd.read_csv(list(uploaded.keys())[0])
print(f"📊 Manifest có {len(manifest)} samples")
print(manifest.head())

📤 Upload file manifest CSV (image_id, image_path, text_gt)


Saving labels_manual.csv to labels_manual.csv
📊 Manifest có 550 samples
     image_id   text_gt
0  plate_0001  51G10096
1  plate_0002  51G10096
2  plate_0003  51A65474
3  plate_0004  51G51936
4  plate_0005  51G51936


## Cách 2: Mount Google Drive (cho dataset lớn)

In [ ]:
# Uncomment nếu data nằm trên Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/vn_plate_data'  # Thay đổi theo path của bạn

# 4. Load Model với Unsloth

In [ ]:
from unsloth import FastVisionModel
import torch

# Load Qwen2-VL-2B-Instruct với 4-bit quantization
model_name = "Qwen/Qwen2-VL-2B-Instruct"

print("⏳ Đang load model...")
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = model_name,
    load_in_4bit = True,  # 4-bit quantization
    use_gradient_checkpointing = "unsloth",  # Tiết kiệm VRAM
)

print("✅ Model loaded!")
print(f"Model: {model_name}")
print(f"Trainable parameters: {model.num_parameters(only_trainable=True) / 1e6:.1f}M")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Đang load model...
==((====))==  Unsloth 2026.5.8: Fast Qwen2_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found
✅ Model loaded!
Model: Qwen/Qwen2-VL-2B-Instruct
Trainable parameters: 0.0M


In [ ]:
# Apply LoRA adapter
model = FastVisionModel.get_peft_model(
    model,
    r = 16,  # LoRA rank
    lora_alpha = 32,  # Scaling factor
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    use_gradient_checkpointing = "unsloth",
)

FastVisionModel.for_inference(model)  # Bật inference mode

print("✅ LoRA adapter applied!")
print(f"Trainable parameters: {model.num_parameters(only_trainable=True) / 1e6:.1f}M")

✅ LoRA adapter applied!
Trainable parameters: 18.5M


# 5. Prepare Dataset

In [ ]:
import os
from PIL import Image
from datasets import Dataset
import pandas as pd

def create_dataset_from_folder(image_dir, manifest_path=None):
    if manifest_path:
        df = pd.read_csv(manifest_path)
        gt_dict = dict(zip(df['image_id'].astype(str), df['text_gt'].astype(str)))
    else:
        gt_dict = {}

    conversations = []
    for img_file in os.listdir(image_dir):
        if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(image_dir, img_file)
            image_id = os.path.splitext(img_file)[0]
            plate_text = str(gt_dict.get(image_id, "UNKNOWN"))

            conv = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": img_path},
                        {"type": "text", "text": "Đọc biển số xe trong ảnh này:"}
                    ]
                },
                {"role": "assistant", "content": plate_text}
            ]
            conversations.append({"messages": conv})

    return Dataset.from_list(conversations)

print("✅ Dataset loader function fixed!")

✅ Dataset loader function fixed!


In [ ]:
# Load dataset
# Thay đổi path theo cách bạn upload data

# Cách 1: Từ manifest CSV
# dataset = load_dataset_from_manifest('/content/manifest.csv')

# Cách 2: Từ folder + manifest
# dataset = create_dataset_from_folder('/content/images', '/content/manifest.csv')

# Cách 3: Upload ảnh trực tiếp
from google.colab import files
import zipfile
import io

print("📤 Upload file ảnh (zip hoặc folder)...")
uploaded = files.upload()

# Giải nén nếu là zip
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(uploaded[filename]), 'r') as zip_ref:
            zip_ref.extractall('/content/vn_plate_data')
        print(f"✅ Đã giải nén {filename}")

# Load dataset
image_dir = '/content/vn_plate_data'
dataset = create_dataset_from_folder(image_dir)

print(f"📊 Dataset loaded: {len(dataset)} samples")
print(f"Example: {dataset[0]}")

📤 Upload file ảnh (zip hoặc folder)...


Saving crops.zip to crops (2).zip
✅ Đã giải nén crops (2).zip
📊 Dataset loaded: 550 samples
Example: {'messages': [{'role': 'user', 'content': [{'type': 'image', 'image': '/content/vn_plate_data/plate_0132.jpg', 'text': None}, {'type': 'text', 'image': None, 'text': 'Đọc biển số xe trong ảnh này:'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'image': None, 'text': 'UNKNOWN'}]}]}


# 6. Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bf16_supported

# 1. Pre-format the dataset to avoid formatting_func issues with Unsloth Qwen2-VL
def apply_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize = False,
        add_generation_prompt = False,
    )
    return {"text": text}

# Map dataset directly
train_dataset = dataset.map(apply_template, num_proc = 4)

# 2. Training arguments
training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 10,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = not is_bf16_supported(),
    bf16 = is_bf16_supported(),
    logging_steps = 1,
    output_dir = "outputs/qwen_vl_finetuned",
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    report_to = "none",
    seed = 42,
)

# 3. Initialize Trainer using the pre-formatted text column
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    args = training_args,
    max_seq_length = 2048,
)

print("✅ Trainer re-configured using pre-formatted dataset mapping!")

Map (num_proc=4):   0%|          | 0/550 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/550 [00:00<?, ? examples/s]

✅ Trainer re-configured using pre-formatted dataset mapping!


In [ ]:
# Bắt đầu training
print("🚀 Bắt đầu training...")
print("=" * 50)

trainer.train()

print("=" * 50)
print("✅ Training hoàn tất!")

🚀 Bắt đầu training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 550 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 2,227,450,368 (0.83% trained)


Step,Training Loss
1,7.099461
2,7.099461
3,6.826442
4,6.153030
5,5.474853
6,4.899398
7,4.344204
8,4.000076
9,3.762539
10,3.606936


Unsloth: Restored added_tokens_decoder metadata in outputs/qwen_vl_finetuned/checkpoint-60/tokenizer_config.json.


✅ Training hoàn tất!


# 7. Save & Push lên Hugging Face

In [ ]:
# Save locally
trainer.save_model("/content/qwen_vl_finetuned")
tokenizer.save_pretrained("/content/qwen_vl_finetuned")
print("💾 Đã save model locally")

Unsloth: Restored added_tokens_decoder metadata in /content/qwen_vl_finetuned/tokenizer_config.json.


💾 Đã save model locally


In [ ]:
from huggingface_hub import HfApi

# Tự động lấy username từ Token để tránh lỗi 403
api = HfApi()
user_info = api.whoami(token = HF_TOKEN)
username = user_info["name"]

REPO_ID = f"{username}/vn-plate-qwen2-vl-2b"

print(f"📤 Đang push lên Hugging Face Repo: {REPO_ID}")

# Push model/adapter
model.push_to_hub(
    REPO_ID,
    token = HF_TOKEN,
    commit_message = "Fine-tune Qwen2-VL-2B for VN plate recognition"
)

# Push tokenizer
tokenizer.push_to_hub(
    REPO_ID,
    token = HF_TOKEN,
    commit_message = "Add tokenizer for Qwen2-VL"
)

print(f"✅ Push thành công!")
print(f"🔗 Link model: https://huggingface.co/{REPO_ID}")

📤 Đang push lên Hugging Face Repo: toanlion24/vn-plate-qwen2-vl-2b


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors: 100%|##########| 73.9MB / 73.9MB            

No files have been modified since last commit. Skipping to prevent empty commit.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp3egbbo3g/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Push thành công!
🔗 Link model: https://huggingface.co/toanlion24/vn-plate-qwen2-vl-2b


# 8. Test Inference

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLProcessor
from qwen_vl_utils import process_vision_info
from PIL import Image
import torch

# Load model vừa train (từ local)
test_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "/content/qwen_vl_finetuned",
    torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map = "auto",
)

test_processor = Qwen2VLProcessor.from_pretrained("/content/qwen_vl_finetuned")

def recognize_plate(image_path, model, processor):
    """Nhận diện biển số từ ảnh"""

    # Load image
    if isinstance(image_path, str):
        image = Image.open(image_path)
    else:
        image = image_path

    # Build conversation
    conversation = [
        {
            "role": "system",
            "content": "Bạn là hệ thống nhận diện biển số xe Việt Nam. Chỉ trả về biển số, không giải thích."
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Đọc biển số:"}
            ]
        }
    ]

    # Process
    text = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(conversation)

    inputs = processor(
        text = [text],
        images = image_inputs,
        videos = video_inputs,
        padding = True,
        return_tensors = "pt",
    ).to(model.device)

    # Generate
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens = 32,
            do_sample = False,
        )

    # Decode
    output = processor.batch_decode(
        generated_ids[:, len(inputs.input_ids[0]):],
        skip_special_tokens = True,
        clean_up_tokenization_spaces = False
    )[0]

    return output.strip()

# Test với ảnh đầu tiên trong dataset
test_img = dataset[0]['messages'][0]['content'][0]['image']
result = recognize_plate(test_img, test_model, test_processor)
print(f"🖼️  Image: {test_img}")
print(f"📝 Predicted: {result}")
print(f"📝 Ground truth: {dataset[0]['messages'][1]['content']}")

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Both `max_new_tokens` (=32) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🖼️  Image: /content/vn_plate_data/plate_0132.jpg
📝 Predicted: 
📝 Ground truth: [{'type': 'text', 'image': None, 'text': 'UNKNOWN'}]


# 9. Download Model về Local

In [4]:
import shutil
from google.colab import files
import os

# Define the model directory path
model_dir = '/content/qwen_vl_finetuned'
output_zip_path = f'{model_dir}.zip'

# Check if the model directory exists
if not os.path.exists(model_dir):
    print(f"❌ Error: Model directory '{model_dir}' not found. Please verify that the model was saved correctly in cell `mde5VtUu6PyO`.")
else:
    # Create zip archive using shutil
    # This creates a zip file at /content/qwen_vl_finetuned.zip
    # and the contents inside the zip will be under a folder named 'qwen_vl_finetuned'
    shutil.make_archive(
        base_name = model_dir, # Output zip name without .zip extension
        format = 'zip',
        root_dir = '/content', # Directory to start archiving from
        base_dir = 'qwen_vl_finetuned' # The specific directory within root_dir to archive
    )

    # Download
    files.download(output_zip_path)

    print("📥 Model đã được download!")
    print("Giải nén vào folder models/vn_plate_qwen2vl/")

❌ Error: Model directory '/content/qwen_vl_finetuned' not found. Please verify that the model was saved correctly in cell `mde5VtUu6PyO`.


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---

## Troubleshooting

### Lỗi CUDA out of memory
```python
# Giảm batch size
per_device_train_batch_size = 1
gradient_accumulation_steps = 8
```

### Lỗi Image not found
```python
# Kiểm tra image path
import os
print(os.path.exists(image_path))
```

### Lỗi HF login failed
```python
# Kiểm tra token
!huggingface-cli whoami
```

---

## Next Steps

1. **Inference trên local**: Sử dụng model đã push lên HF
2. **Đánh giá**: Chạy eval trên test set
3. **Tuning**: Thử các hyperparameters khác

**Model URL**: https://huggingface.co/{REPO_ID}
